# 02 — Statistical Analysis

In-depth statistical analysis of F1 data:
1. Tire degradation curves
2. Lap time significance tests
3. Qualifying vs race performance
4. Telemetry clustering (driving styles)
5. Pit strategy analysis

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data import FastF1Loader, JolpicaClient
from src.data.preprocessing import F1Preprocessor
from src.analysis import LapAnalyzer, TelemetryAnalyzer, RaceAnalyzer

sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Tire Degradation Curves

Model lap time vs. tire age per compound. Fit linear and polynomial models
to quantify degradation rates.

In [ ]:
loader = FastF1Loader()
analyzer = LapAnalyzer()

# Load a race with clear tire degradation (e.g. Barcelona)
session = loader.get_session(2024, 'Spain', 'R')
laps = session.laps
clean = analyzer.filter_representative_laps(laps)

print(f'Compounds used: {clean["Compound"].unique()}')
print(f'Clean laps: {len(clean)}')

In [ ]:
# Fit degradation curves per compound
deg_results = analyzer.degradation_by_compound(clean, degree=2)

fig, axes = plt.subplots(1, len(deg_results), figsize=(6 * len(deg_results), 5))
if len(deg_results) == 1:
    axes = [axes]

for ax, (compound, result) in zip(axes, deg_results.items()):
    compound_laps = clean[clean['Compound'] == compound]
    analyzer.plot_degradation(compound_laps, compound=compound, ax=ax)

plt.suptitle('Tire Degradation by Compound', fontsize=14)
plt.tight_layout()
plt.show()

# Print R² scores
for compound, result in deg_results.items():
    print(f'{compound}: R²={result["r_squared"]:.4f}, coefficients={result["coefficients"]}')

## 2. Lap Time Significance Tests

In [ ]:
# Kruskal-Wallis test across all drivers
kw_result = analyzer.kruskal_wallis_test(clean, group_col='Driver')
print(f'Kruskal-Wallis H={kw_result["H_statistic"]:.2f}, p={kw_result["p_value"]:.2e}')
print(f'Significant at α=0.05: {kw_result["significant_005"]}')

In [ ]:
# Pairwise Mann-Whitney tests for top 5 drivers
summary = analyzer.compare_distributions(clean)
top5 = summary.head(5).index.tolist()

print('Pairwise Mann-Whitney U tests (top 5 by median):')
for i in range(len(top5)):
    for j in range(i + 1, len(top5)):
        a = clean[clean['Driver'] == top5[i]]['LapTime']
        b = clean[clean['Driver'] == top5[j]]['LapTime']
        result = analyzer.mann_whitney_test(a, b)
        sig = '***' if result['p_value'] < 0.001 else ('**' if result['p_value'] < 0.01 else ('*' if result['p_value'] < 0.05 else 'ns'))
        print(f'  {top5[i]} vs {top5[j]}: U={result["U_statistic"]:.0f}, p={result["p_value"]:.4f} {sig}')

## 3. Qualifying vs Race Performance

In [ ]:
jolpica = JolpicaClient()

# Load multi-season results
results = jolpica.get_all_results(start_year=2018, end_year=2024)
print(f'Total results: {len(results)}')

# Overall correlation
race_analyzer = RaceAnalyzer()
corr = race_analyzer.qualifying_race_correlation(results)
print(f'\nOverall grid-finish correlation:')
print(f'  Spearman r = {corr["spearman_r"]:.4f} (p = {corr["spearman_p"]:.2e})')
print(f'  Pearson r  = {corr["pearson_r"]:.4f}')

In [ ]:
# Per-circuit overtaking difficulty
circuit_corr = race_analyzer.correlation_by_circuit(results)
print('\nTop 10 hardest circuits to overtake (highest grid-finish correlation):')
print(circuit_corr[['circuitId', 'spearman_r', 'avg_position_change']].head(10).to_string(index=False))

fig = race_analyzer.plot_overtaking_difficulty(results, top_n=15)
plt.show()

## 4. Telemetry Clustering

In [ ]:
tel_analyzer = TelemetryAnalyzer()

# Extract telemetry features for all drivers
feature_df = tel_analyzer.extract_features_for_session(laps)

# Aggregate per driver (mean features across laps)
numeric_cols = feature_df.select_dtypes(include=[np.number]).columns.tolist()
driver_features = feature_df.groupby('Driver')[numeric_cols].mean().reset_index()
print(f'Driver features shape: {driver_features.shape}')
driver_features.head()

In [ ]:
# Cluster and visualize
clustered = tel_analyzer.cluster_drivers(driver_features, n_clusters=4)
fig = tel_analyzer.plot_driver_clusters(clustered, method='pca')
plt.show()

# Print cluster assignments
for cluster_id in sorted(clustered['cluster'].unique()):
    drivers = clustered[clustered['cluster'] == cluster_id]['Driver'].tolist()
    print(f'Cluster {cluster_id}: {drivers}')

## 5. Pit Strategy Analysis

In [ ]:
# Analyze pit strategy effectiveness across 2024
results_2024 = jolpica.get_race_results(2024)

# Get pit stops for each round
all_pits = []
for rnd in results_2024['round'].unique():
    try:
        pits = jolpica.get_pit_stops(2024, rnd)
        all_pits.append(pits)
    except Exception:
        continue

if all_pits:
    pit_df = pd.concat(all_pits, ignore_index=True)
    strategy = race_analyzer.analyze_pit_strategy(pit_df, results_2024)
    fig = race_analyzer.plot_strategy_outcomes(strategy)
    plt.show()